In [ ]:
!git clone https://github.com/venkatsaikondra/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification.git

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import timm  # Library for ViT models
import matplotlib.pyplot as plt
import numpy as np

# Paths based on your previous split
data_dir = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Train_Test_SPlit_ViT_Data_CLAHE"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485], std=[0.229]) # Grayscale/CLAHE normalization
])

train_ds = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
val_ds = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
test_ds = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

In [ ]:
class ViTResNetHybrid(nn.Module):
    def __init__(self, num_classes=4):
        super(ViTResNetHybrid, self).__init__()
        # 1. ResNet Backbone (Feature Extractor)
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) # Remove FC and Pooling

        # 2. ViT Component (using timm)
        # We use a 'vit_base_patch16_224' or similar
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)

        # 3. Bridge: Project ResNet features to ViT embedding dimension
        self.projector = nn.Conv2d(2048, 768, kernel_size=1)

        # 4. Final Classification Head
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, x):
        # Extract spatial features via ResNet
        features = self.backbone(x) # Shape: [batch, 2048, 7, 7]

        # Project and flatten for Transformer
        x = self.projector(features) # [batch, 768, 7, 7]
        x = x.flatten(2).transpose(1, 2) # [batch, 49, 768]

        # Pass through ViT blocks
        x = self.vit.blocks(x)
        x = self.vit.norm(x)

        # Global Average Pooling and Classify
        x = x.mean(dim=1)
        return self.classifier(x)

model = ViTResNetHybrid(num_classes=4).to('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    # Metrics
    print(classification_report(y_true, y_pred, target_names=train_ds.classes))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=train_ds.classes, yticklabels=train_ds.classes)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# We target the last layer of the ResNet backbone before the bridge
target_layers = [model.backbone[-1]]

def generate_explainability(input_tensor, original_image):
    cam = GradCAM(model=model, target_layers=target_layers)

    # Generate heatmap
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(0)])
    grayscale_cam = grayscale_cam[0, :]

    # Overlay on image
    visualization = show_cam_on_image(original_image, grayscale_cam, use_rgb=True)
    plt.imshow(visualization)
    plt.title("Grad-CAM Explainability")
    plt.show()